# Capítulo 11: La Latencia Mata al Modelo (Optimización para Producción)

## 🎯 Objetivo del Notebook

Este notebook demuestra técnicas de optimización de modelos para producción:
- Benchmarking de modelos
- Cuantización (INT8, FP16)
- Pruning (poda de redes neuronales)
- Exportación a ONNX
- Comparación de rendimiento
- Análisis ético de la optimización

> *"Tu modelo es un Ferrari, pero si tarda 2 segundos en arrancar, nadie lo compra."*

In [ ]:
# Celda 1: Importaciones

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import time
import os
from functools import wraps

# Para ONNX
import onnx
import onnxruntime as ort

# Configuración
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar disponibilidad de GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")

## 📊 Celda 2: Modelo Base

Crearemos un modelo CNN para clasificación de imágenes (similar a CIFAR-10)
que usaremos como base para todas las optimizaciones.

In [ ]:
# Celda 2: Modelo base

class ModeloCNN(nn.Module):
    """
    CNN para clasificación de imágenes.
    Arquitectura similar a VGG pero más ligera.
    """
    
    def __init__(self, num_classes=10):
        super(ModeloCNN, self).__init__()
        
        # Capas convolucionales
        self.features = nn.Sequential(
            # Bloque 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Bloque 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Bloque 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        
        # Capas fully connected
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


def contar_parametros(model):
    """Cuenta el número total de parámetros entrenables."""
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params


# Crear modelo
model = ModeloCNN(num_classes=10).to(device)
model.eval()  # Modo evaluación

total_params = contar_parametros(model)
print(f"Modelo creado exitosamente")
print(f"Parámetros totales: {total_params:,}")
print(f"Tamaño estimado: {total_params * 4 / 1e6:.2f} MB (FP32)")
print(f"\nArquitectura:")
print(model)

## ⏱️ Celda 3: Benchmarking

Mediremos el rendimiento del modelo base bajo diferentes condiciones.
El benchmarking es como **cronometrar a un mesonero**: ¿cuánto tarda en traer tu pedido?

In [ ]:
# Celda 3: Benchmarking

def benchmark_inferencia(model, input_tensor, num_runs=100, warmup=10):
    """
    Realiza benchmark de inferencia.
    
    Args:
        model: Modelo a evaluar
        input_tensor: Tensor de entrada
        num_runs: Número de ejecuciones para el benchmark
        warmup: Ejecuciones de calentamiento
    
    Returns:
        Diccionario con métricas de rendimiento
    """
    model.eval()
    
    # Warmup (como calentar antes de un partido)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_tensor)
    
    # Benchmark
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = model(input_tensor)
            end = time.perf_counter()
            times.append((end - start) * 1000)  # Convertir a ms
    
    times = np.array(times)
    
    # Calcular métricas
    metrics = {
        'promedio_ms': np.mean(times),
        'mediana_ms': np.median(times),
        'p95_ms': np.percentile(times, 95),
        'p99_ms': np.percentile(times, 99),
        'std_ms': np.std(times),
        'min_ms': np.min(times),
        'max_ms': np.max(times),
        'throughput': 1000 / np.mean(times)  # Peticiones por segundo
    }
    
    return metrics, times


def contar_parametros(model):
    """Cuenta el número total de parámetros entrenables."""
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params


def get_model_size_mb(model, path='temp_model.pth'):
    """Calcula el tamaño del modelo en MB."""
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return size_mb


# Crear tensor de entrada de ejemplo (batch de 1, 3 canales, 32x32 píxeles)
input_tensor = torch.randn(1, 3, 32, 32).to(device)

# Benchmark del modelo base
print("\n" + "="*60)
print("BENCHMARKING DEL MODELO BASE")
print("="*60)

metrics_base, times_base = benchmark_inferencia(model, input_tensor, num_runs=100)
size_base = get_model_size_mb(model)
params_base = contar_parametros(model)

print(f"\nMétricas de Rendimiento:")
print(f"- Promedio: {metrics_base['promedio_ms']:.2f} ms")
print(f"- Mediana: {metrics_base['mediana_ms']:.2f} ms")
print(f"- P95: {metrics_base['p95_ms']:.2f} ms")
print(f"- P99: {metrics_base['p99_ms']:.2f} ms")
print(f"- Throughput: {metrics_base['throughput']:.2f} peticiones/segundo")
print(f"\nMétricas de Modelo:")
print(f"- Parámetros: {params_base:,}")
print(f"- Tamaño: {size_base:.2f} MB")

# Visualizar distribución de tiempos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(times_base, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(metrics_base['promedio_ms'], color='red', linestyle='--', 
                label=f"Promedio: {metrics_base['promedio_ms']:.2f}ms")
axes[0].axvline(metrics_base['p95_ms'], color='orange', linestyle='--', 
                label=f"P95: {metrics_base['p95_ms']:.2f}ms")
axes[0].set_xlabel('Tiempo de inferencia (ms)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Tiempos de Inferencia - Modelo Base')
axes[0].legend()

# Boxplot
axes[1].boxplot(times_base, vert=True)
axes[1].set_ylabel('Tiempo de inferencia (ms)')
axes[1].set_title('Boxplot de Tiempos de Inferencia')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print("⚡ El mesonero promedio tarda {:.2f}ms en traer tu pedido".format(metrics_base['promedio_ms']))
print(f"{'='*60}")

## 🗜️ Celda 4: Cuantización

La cuantización es como **comprimir una foto sin perder calidad visible**.
Reducimos la precisión de 32 bits a 8 bits, logrando:
- 75% de reducción de tamaño
- 3-4x más rápido en hardware compatible
- Pérdida mínima de precisión (< 2%)

In [ ]:
# Celda 4: Cuantización

def cuantizar_modelo(model, dtype=torch.qint8):
    """
    Aplica cuantización dinámica a un modelo.
    
    Args:
        model: Modelo PyTorch
        dtype: Tipo de dato destino (qint8, qint16, qfloat16)
    
    Returns:
        Modelo cuantizado
    """
    model.eval()
    
    # Copiar modelo para no modificar el original
    model_cuantizado = ModeloCNN(num_classes=10).to(device)
    model_cuantizado.load_state_dict(model.state_dict())
    
    # Aplicar cuantización dinámica
    model_cuantizado = torch.quantization.quantize_dynamic(
        model_cuantizado,
        {nn.Linear},  # Cuantizar capas lineales
        dtype=dtype
    )
    
    return model_cuantizado


def contar_parametros(model):
    """Cuenta el número total de parámetros entrenables."""
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params


def get_model_size_mb(model, path='temp_model.pth'):
    """Calcula el tamaño del modelo en MB."""
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return size_mb


# Aplicar cuantización INT8
print("\n" + "="*60)
print("CUANTIZACIÓN INT8")
print("="*60)

model_int8 = cuantizar_modelo(model, dtype=torch.qint8)
model_int8.eval()

# Benchmark del modelo cuantizado
metrics_int8, times_int8 = benchmark_inferencia(model_int8, input_tensor, num_runs=100)
size_int8 = get_model_size_mb(model_int8)
params_int8 = contar_parametros(model_int8)

print(f"\nMétricas de Modelo Cuantizado INT8:")
print(f"- Parámetros: {params_int8:,}")
print(f"- Tamaño: {size_int8:.2f} MB")
print(f"- Reducción de tamaño: {(1 - size_int8/size_base)*100:.1f}%")

print(f"\nMétricas de Rendimiento:")
print(f"- Promedio: {metrics_int8['promedio_ms']:.2f} ms")
print(f"- P95: {metrics_int8['p95_ms']:.2f} ms")
print(f"- Throughput: {metrics_int8['throughput']:.2f} peticiones/segundo")
print(f"- Speedup: {metrics_base['promedio_ms']/metrics_int8['promedio_ms']:.2f}x más rápido")

# Comparación visual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comparación de tiempos
modelos = ['Base (FP32)', 'Cuantizado (INT8)']
tiempos = [metrics_base['promedio_ms'], metrics_int8['promedio_ms']]
colores = ['steelblue', 'coral']

bars = axes[0].bar(modelos, tiempos, color=colores, edgecolor='black')
axes[0].set_ylabel('Tiempo promedio (ms)')
axes[0].set_title('Comparación de Tiempos de Inferencia')
for bar, tiempo in zip(bars, tiempos):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{tiempo:.1f}ms', ha='center', va='bottom', fontweight='bold')

# Comparación de tamaños
tamanios = [size_base, size_int8]
bars2 = axes[1].bar(modelos, tamanios, color=colores, edgecolor='black')
axes[1].set_ylabel('Tamaño del modelo (MB)')
axes[1].set_title('Comparación de Tamaño de Modelo')
for bar, tam in zip(bars2, tamanios):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{tam:.1f}MB', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Simular pérdida de precisión (en producción, esto se mediría con datos reales)
precision_base = 94.5
precision_int8 = 93.8  # Pérdida típica de 0.5-1%

print(f"\n{'='*60}
print("📊 RESULTADOS CUANTIZACIÓN INT8:")
print(f"{'='*60}")
print(f"✅ Velocidad: {metrics_base['promedio_ms']/metrics_int8['promedio_ms']:.2f}x más rápido")
print(f"✅ Tamaño: {(1 - size_int8/size_base)*100:.1f}% más pequeño")
print(f"⚠️  Precisión: {precision_base}% → {precision_int8}% (-{precision_base-precision_int8:.1f}%)")

## ✂️ Celda 5: Pruning (Poda)

El pruning elimina conexiones innecesarias, como **podar un árbol para que crezca mejor**.
Según Han et al. (2016), muchos modelos tienen >90% de redundancia.

In [ ]:
# Celda 5: Pruning

def aplicar_pruning(model, amount=0.3, tipo='unstructured'):
    """
    Aplica pruning al modelo.
    
    Args:
        model: Modelo a podar
        amount: Proporción de pesos a eliminar (0.0 a 1.0)
        tipo: 'unstructured' o 'structured'
    
    Returns:
        Modelo podado
    """
    model.eval()
    
    # Copiar modelo
    model_podado = ModeloCNN(num_classes=10).to(device)
    model_podado.load_state_dict(model.state_dict())
    
    if tipo == 'unstructured':
        # Pruning no estructurado: elimina pesos individuales
        for name, module in model_podado.named_modules():
            if isinstance(module, (nn.Linear, nn.Conv2d)):
                prune.l1_unstructured(module, name='weight', amount=amount)
    
    elif tipo == 'structured':
        # Pruning estructurado: elimina neuronas completas
        for name, module in model_podado.named_modules():
            if isinstance(module, nn.Linear):
                prune.ln_structured(
                    module, 
                    name='weight', 
                    amount=amount,
                    n=2,  # Norma L2
                    dim=0  # A lo largo de las filas (neuronas)
                )
            elif isinstance(module, nn.Conv2d):
                prune.ln_structured(
                    module,
                    name='weight',
                    amount=amount,
                    n=2,
                    dim=0
                )
    
    return model_podado


def contar_parametros(model):
    """Cuenta el número total de parámetros entrenables."""
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params


def get_model_size_mb(model, path='temp_model.pth'):
    """Calcula el tamaño del modelo en MB."""
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return size_mb


# Aplicar pruning con diferentes niveles
print("\n" + "="*60)
print("PRUNING (PODA DE REDES NEURONALES)")
print("="*60)

niveles_pruning = [0.3, 0.5, 0.7]
resultados_pruning = []

for nivel in niveles_pruning:
    print(f"\n🔄 Pruning al {int(nivel*100)}%...")
    
    model_podado = aplicar_pruning(model, amount=nivel, tipo='unstructured')
    model_podado.eval()
    
    # Benchmark
    metrics_podado, times_podado = benchmark_inferencia(model_podado, input_tensor, num_runs=100)
    size_podado = get_model_size_mb(model_podado)
    params_podado = contar_parametros(model_podado)
    
    # Calcular sparsity (porcentaje de ceros)
    total_weights = 0
    zero_weights = 0
    for module in model_podado.modules():
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            total_weights += module.weight.nelement()
            zero_weights += torch.sum(module.weight == 0).item()
    
    sparsity = zero_weights / total_weights * 100
    
    resultado = {
        'nivel': f"{int(nivel*100)}%",
        'sparsity': f"{sparsity:.1f}%",
        'params': params_podado,
        'size_mb': size_podado,
        'latencia_ms': metrics_podado['promedio_ms'],
        'speedup': metrics_base['promedio_ms'] / metrics_podado['promedio_ms']
    }
    resultados_pruning.append(resultado)
    
    print(f"  - Sparsity: {sparsity:.1f}%")
    print(f"  - Parámetros: {params_podado:,}")
    print(f"  - Tamaño: {size_podado:.2f} MB")
    print(f"  - Latencia: {metrics_podado['promedio_ms']:.2f} ms")
    print(f"  - Speedup: {metrics_podado['speedup']:.2f}x")

# Crear DataFrame de resultados
df_pruning = pd.DataFrame(resultados_pruning)
print("\n" + "="*60)
print("📊 RESUMEN DE PRUNING")
print("="*60)
print(df_pruning.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

niveles = df_pruning['nivel'].tolist()
latencias = df_pruning['latencia_ms'].tolist()
tamanios = df_pruning['size_mb'].tolist()
speedups = df_pruning['speedup'].tolist()

axes[0].bar(niveles, latencias, color='coral', edgecolor='black')
axes[0].set_xlabel('Nivel de Pruning')
axes[0].set_ylabel('Latencia (ms)')
axes[0].set_title('Latencia vs Nivel de Pruning')

axes[1].bar(niveles, tamanios, color='lightgreen', edgecolor='black')
axes[1].set_xlabel('Nivel de Pruning')
axes[1].set_ylabel('Tamaño (MB)')
axes[1].set_title('Tamaño vs Nivel de Pruning')

axes[2].bar(niveles, speedups, color='skyblue', edgecolor='black')
axes[2].set_xlabel('Nivel de Pruning')
axes[2].set_ylabel('Speedup (x)')
axes[2].set_title('Speedup vs Nivel de Pruning')
axes[2].axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Baseline')
axes[2].legend()

plt.tight_layout()
plt.show()

## 🔄 Celda 6: Exportación ONNX

ONNX es el **adaptador universal de cargadores** para modelos de ML.
Permite exportar modelos de PyTorch, TensorFlow, etc. a un formato estándar.

In [ ]:
# Celda 6: Exportación ONNX

def exportar_a_onnx(model, input_shape, output_path, dynamic_batch=True):
    """
    Exporta un modelo PyTorch a ONNX.
    
    Args:
        model: Modelo PyTorch
        input_shape: Forma del tensor de entrada
        output_path: Ruta del archivo ONNX de salida
        dynamic_batch: Si permitir batch size dinámico
    """
    model.eval()
    
    # Crear tensor de entrada dummy
    dummy_input = torch.randn(*input_shape).to(device)
    
    # Configurar ejes dinámicos
    dynamic_axes = None
    if dynamic_batch:
        dynamic_axes = {
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        }
    
    # Exportar
    torch.onnx.export(
        model,
        dummy_input,
        output_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes=dynamic_axes
    )
    
    print(f"✅ Modelo exportado a: {output_path}")
    print(f"   Tamaño: {os.path.getsize(output_path) / 1e6:.2f} MB")


def benchmark_onnx(session, input_data, num_runs=100, warmup=10):
    """
    Benchmark de inferencia ONNX Runtime.
    """
    # Warmup
    for _ in range(warmup):
        _ = session.run(None, {'input': input_data})
    
    # Benchmark
    times = []
    for _ in range(num_runs):
        start = time.perf_counter()
        _ = session.run(None, {'input': input_data})
        end = time.perf_counter()
        times.append((end - start) * 1000)
    
    times = np.array(times)
    
    return {
        'promedio_ms': np.mean(times),
        'p95_ms': np.percentile(times, 95),
        'throughput': 1000 / np.mean(times)
    }


# Exportar modelo base a ONNX
print("\n" + "="*60)
print("EXPORTACIÓN A ONNX")
print("="*60)

# Preparar modelo para exportación
model.eval()
dummy_input = torch.randn(1, 3, 32, 32).to(device)

# Exportar modelo base
output_path = 'model_base.onnx'
torch.onnx.export(
    model,
    dummy_input,
    output_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

size_onnx = os.path.getsize(output_path) / 1e6
print(f"✅ Modelo base exportado: {output_path}")
print(f"   Tamaño ONNX: {size_onnx:.2f} MB")

# Verificar modelo ONNX
model_onnx = onnx.load(output_path)
onnx.checker.check_model(model_onnx)
print(f"✅ Modelo ONNX verificado exitosamente")

# Crear sesión ONNX Runtime
print(f"\n🔧 Creando sesión ONNX Runtime...")
session = ort.InferenceSession(output_path)
print(f"✅ Sesión creada")
print(f"   Provider: {session.get_providers()}")
print(f"   Input: {session.get_inputs()[0].name}")
print(f"   Output: {session.get_outputs()[0].name}")

# Benchmark ONNX Runtime
input_numpy = dummy_input.cpu().numpy()
metrics_onnx = benchmark_onnx(session, input_numpy, num_runs=100)

print(f"\n📊 Benchmark ONNX Runtime:")
print(f"- Promedio: {metrics_onnx['promedio_ms']:.2f} ms")
print(f"- P95: {metrics_onnx['p95_ms']:.2f} ms")
print(f"- Throughput: {metrics_onnx['throughput']:.2f} peticiones/segundo")
print(f"- Speedup vs PyTorch: {metrics_base['promedio_ms']/metrics_onnx['promedio_ms']:.2f}x")

# Visualización comparativa
fig, ax = plt.subplots(figsize=(10, 5))

modelos = ['PyTorch FP32', 'ONNX Runtime']
tiempos = [metrics_base['promedio_ms'], metrics_onnx['promedio_ms']]
colores = ['steelblue', 'lightgreen']

bars = ax.bar(modelos, tiempos, color=colores, edgecolor='black')
ax.set_ylabel('Tiempo promedio (ms)')
ax.set_title('Comparación: PyTorch vs ONNX Runtime')

for bar, tiempo in zip(bars, tiempos):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{tiempo:.1f}ms', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n💡 ONNX Runtime puede ser más rápido por las optimizaciones automáticas")

## 📈 Celda 7: Comparación de Rendimiento

Comparamos todas las técnicas de optimización en un solo gráfico.
Esto es como **comparar diferentes autos en una pista de pruebas**.

In [ ]:
# Celda 7: Comparación de rendimiento

def contar_parametros(model):
    """Cuenta el número total de parámetros entrenables."""
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params


def get_model_size_mb(model, path='temp_model.pth'):
    """Calcula el tamaño del modelo en MB."""
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return size_mb


# Preparar datos de comparación
print("\n" + "="*70)
print("📊 COMPARACIÓN COMPLETA DE OPTIMIZACIONES")
print("="*70)

# Datos del modelo base
datos_comparacion = [{
    'Modelo': 'Base (FP32)',
    'Framework': 'PyTorch',
    'Latencia (ms)': metrics_base['promedio_ms'],
    'P95 (ms)': metrics_base['p95_ms'],
    'Tamaño (MB)': size_base,
    'Throughput': metrics_base['throughput'],
    'Speedup': 1.0
}]

# Datos del modelo INT8
datos_comparacion.append({
    'Modelo': 'Cuantizado INT8',
    'Framework': 'PyTorch',
    'Latencia (ms)': metrics_int8['promedio_ms'],
    'P95 (ms)': metrics_int8['p95_ms'],
    'Tamaño (MB)': size_int8,
    'Throughput': metrics_int8['throughput'],
    'Speedup': metrics_base['promedio_ms'] / metrics_int8['promedio_ms']
})

# Datos del modelo ONNX
datos_comparacion.append({
    'Modelo': 'ONNX Runtime',
    'Framework': 'ONNX',
    'Latencia (ms)': metrics_onnx['promedio_ms'],
    'P95 (ms)': metrics_onnx['p95_ms'],
    'Tamaño (MB)': size_onnx,
    'Throughput': metrics_onnx['throughput'],
    'Speedup': metrics_base['promedio_ms'] / metrics_onnx['promedio_ms']
})

# Crear DataFrame
df_comparacion = pd.DataFrame(datos_comparacion)
print("\n", df_comparacion.to_string(index=False))

# Visualización completa
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

modelos = df_comparacion['Modelo'].tolist()

# 1. Latencia
ax1 = axes[0, 0]
latencias = df_comparacion['Latencia (ms)'].tolist()
colors1 = ['steelblue', 'coral', 'lightgreen']
bars1 = ax1.bar(modelos, latencias, color=colors1, edgecolor='black')
ax1.set_ylabel('Latencia (ms)')
ax1.set_title('⏱️ Latencia de Inferencia')
for bar, val in zip(bars1, latencias):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}ms', ha='center', va='bottom', fontweight='bold')

# 2. Tamaño
ax2 = axes[0, 1]
tamanios = df_comparacion['Tamaño (MB)'].tolist()
bars2 = ax2.bar(modelos, tamanios, color=colors1, edgecolor='black')
ax2.set_ylabel('Tamaño (MB)')
ax2.set_title('📦 Tamaño del Modelo')
for bar, val in zip(bars2, tamanios):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}MB', ha='center', va='bottom', fontweight='bold')

# 3. Throughput
ax3 = axes[1, 0]
throughputs = df_comparacion['Throughput'].tolist()
bars3 = ax3.bar(modelos, throughputs, color=colors1, edgecolor='black')
ax3.set_ylabel('Peticiones/segundo')
ax3.set_title('🚀 Throughput')
for bar, val in zip(bars3, throughputs):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold')

# 4. Speedup
ax4 = axes[1, 1]
speedups = df_comparacion['Speedup'].tolist()
bars4 = ax4.bar(modelos, speedups, color=colors1, edgecolor='black')
ax4.set_ylabel('Speedup (x)')
ax4.set_title('⚡ Speedup vs Base')
ax4.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Baseline')
for bar, val in zip(bars4, speedups):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}x', ha='center', va='bottom', fontweight='bold')
ax4.legend()

plt.suptitle('📊 Comparación de Optimizaciones de Modelo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Resumen ejecutivo
print("\n" + "="*70)
print("🎯 RESUMEN EJECUTIVO")
print("="*70)
print(f"✅ Mejor velocidad: ONNX Runtime ({metrics_onnx['promedio_ms']:.1f}ms)")
print(f"✅ Mejor compresión: Cuantizado INT8 ({(1-size_int8/size_base)*100:.1f}% reducción)")
print(f"✅ Mejor throughput: ONNX Runtime ({metrics_onnx['throughput']:.1f} peticiones/seg)")
print(f"\n💡 Recomendación: Para producción, usa ONNX Runtime + Cuantización INT8")

## ⚖️ Celda 8: Ética - ¿Cuánta Precisión Sacrificas?

La optimización siempre implica un trade-off.
**¿Cuánta precisión estás dispuesto a sacrificar por velocidad?**

Esta es una pregunta ética fundamental en producción de ML.

In [ ]:
# Celda 8: Ética - ¿cuánta precisión sacrificas?

def contar_parametros(model):
    """Cuenta el número total de parámetros entrenables."""
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params


print("\n" + "="*70)
print("⚖️  ANÁLISIS ÉTICO DE OPTIMIZACIÓN")
print("="*70)

# Definir escenarios éticos
escenarios = [
    {
        'nombre': 'Clasificación de imágenes (red social)',
        'precision_base': 94.5,
        'precision_optimizada': 93.8,
        'impacto': 'Bajo - errores no críticos',
        'riesgo': 2,
        'recomendacion': '✅ Optimizar agresivamente'
    },
    {
        'nombre': 'Detección de fraude bancario',
        'precision_base': 96.2,
        'precision_optimizada': 94.8,
        'impacto': 'Medio - pérdidas financieras',
        'riesgo': 5,
        'recomendacion': '⚠️  Optimizar con precaución'
    },
    {
        'nombre': 'Diagnóstico médico asistido',
        'precision_base': 98.1,
        'precision_optimizada': 96.5,
        'impacto': 'Alto - vidas humanas en riesgo',
        'riesgo': 9,
        'recomendacion': '❌ No optimizar sin validación clínica'
    },
    {
        'nombre': 'Reconocimiento facial (vigilancia)',
        'precision_base': 97.3,
        'precision_optimizada': 95.1,
        'impacto': 'Crítico - errores = falsos positivos/negativos',
        'riesgo': 10,
        'recomendacion': '❌ No optimizar - riesgo de sesgo'
    },
    {
        'nombre': 'Recomendación de productos',
        'precision_base': 89.2,
        'precision_optimizada': 87.5,
        'impacto': 'Muy bajo - experiencia de usuario',
        'riesgo': 1,
        'recomendacion': '✅ Optimizar agresivamente'
    }
]

# Crear DataFrame
df_etica = pd.DataFrame(escenarios)

# Mostrar tabla de análisis
print("\n📊 ANÁLISIS POR ESCENARIO:")
print("-" * 70)
for idx, escenario in enumerate(escenarios, 1):
    print(f"\n{idx}. {escenario['nombre']}")
    print(f"   Precisión base: {escenario['precision_base']}%")
    print(f"   Precisión optimizada: {escenario['precision_optimizada']}%")
    print(f"   Pérdida: {escenario['precision_base'] - escenario['precision_optimizada']:.1f}%")
    print(f"   Impacto: {escenario['impacto']}")
    print(f"   Nivel de riesgo: {escenario['riesgo']}/10")
    print(f"   Recomendación: {escenario['recomendacion']}")

# Visualización del análisis ético
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Pérdida de precisión vs Riesgo
nombres_cortos = ['Red Social', 'Fraude', 'Médico', 'Vigilancia', 'Productos']
perdidas = [e['precision_base'] - e['precision_optimizada'] for e in escenarios]
riesgos = [e['riesgo'] for e in escenarios]
colores = ['green' if r <= 3 else 'orange' if r <= 6 else 'red' for r in riesgos]

scatter = axes[0].scatter(perdidas, riesgos, c=colores, s=200, edgecolors='black')
for i, nombre in enumerate(nombres_cortos):
    axes[0].annotate(nombre, (perdidas[i], riesgos[i]), 
                     xytext=(5, 5), textcoords='offset points')

axes[0].set_xlabel('Pérdida de Precisión (%)')
axes[0].set_ylabel('Nivel de Riesgo (1-10)')
axes[0].set_title('⚖️  Pérdida de Precisión vs Riesgo Ético')
axes[0].axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='Umbral de precaución')
axes[0].axhline(y=8, color='red', linestyle='--', alpha=0.5, label='Umbral de alto riesgo')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Niveles de optimización recomendados
categorias = ['Optimizar\nagresivamente', 'Optimizar\ncon precaución', 'No\noptimizar']
contadores = [0, 0, 0]
for e in escenarios:
    if e['riesgo'] <= 3:
        contadores[0] += 1
    elif e['riesgo'] <= 6:
        contadores[1] += 1
    else:
        contadores[2] += 1

colors2 = ['green', 'orange', 'red']
bars = axes[1].bar(categorias, contadores, color=colors2, edgecolor='black')
axes[1].set_ylabel('Número de Escenarios')
axes[1].set_title('📊 Distribución de Recomendaciones Éticas')
for bar, count in zip(bars, contadores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

# Framework de decisión
print("\n" + "="*70)
print("📋 FRAMEWORK DE DECISIÓN ÉTICA")
print("="*70)
print("\nAntes de optimizar, pregúntate:")
print("\n1. ¿La pérdida de precisión afecta vidas humanas?")
print("   → Si SÍ: No optimizar sin validación externa")
print("   → Si NO: Continuar al paso 2")
print("\n2. ¿El sistema tiene sesgos conocidos en grupos subrepresentados?")
print("   → Si SÍ: Evaluar impacto diferencial de la optimización")
print("   → Si NO: Continuar al paso 3")
print("\n3. ¿Hay regulación específica para este dominio?")
print("   → Si SÍ: Cumplir con estándares mínimos de precisión")
print("   → Si NO: Continuar al paso 4")
print("\n4. ¿Los usuarios son conscientes de la optimización?")
print("   → Si NO: Transparentar el proceso de optimización")
print("   → Si SÍ: Documentar trade-offs para auditoría")

# Caso de estudio
print("\n" + "="*70)
print("📝 CASO DE ESTUDIO: Diagnóstico Médico")
print("="*70)
print("\nEscenario: Modelo para detectar neumonía en radiografías")
print("\n• Precisión base: 98.1% (100 radiografías mal diagnosticadas de 10,000)")
print("• Precisión optimizada: 96.5% (350 radiografías mal diagnosticadas de 10,000)")
print("• Diferencia: 250 diagnósticos errados adicionales")
print("\n⚠️  Pregunta ética: ¿Vale la pena la optimización?")
print("\nRespuesta: NO, a menos que:")
print("1. Se valide con ensayos clínicos controlados")
print("2. Se use como apoyo (no reemplazo) del médico")
print("3. Se transparente la limitación a los usuarios")
print("4. Se tenga sistema de fallback para casos dudosos")

print("\n" + "="*70)
print("💡 CONCLUSIÓN ÉTICA")
print("="*70)
print("\nLa optimización es una herramienta poderosa, pero no éticamente neutral.")
print("Siempre debemos preguntarnos:")
print("\n   '¿A quién perjudica esta optimización?'\")
print("\nLa velocidad sin ética es peligrosa.")
print("La ética sin velocidad es inútil en producción.")
print("El equilibrio es la clave.")